# Pandas Data Wrangling for Machine Learning
Accompanies **Appendix B: Pandas Data Wrangling for Machine Learning**.

Pandas stores tabular data columnarly via its internal `BlockManager`. 
This notebook demonstrates downcasting dtypes to cut memory usage by 70%, leak-free grouped imputation, and production dummy schema alignment.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_rows = 15_000

df = pd.DataFrame({
    'CustomerID': np.random.randint(1000, 1800, size=n_rows),
    'State': np.random.choice(['CA', 'NY', 'TX', 'FL', 'IL'], size=n_rows),
    'Amount': [f"${x:,.2f}" if np.random.rand() > 0.04 else "MISSING" for x in np.random.exponential(scale=1200, size=n_rows)],
    'Score': np.random.randint(300, 850, size=n_rows).astype(np.int64)
})

mem_raw = df.memory_usage(deep=True).sum() / 1024
print(f"Raw DataFrame RAM: {mem_raw:.1f} KB")

### Downcasting and Type Optimization
- Strip string characters from numeric columns and coerce to `float32`.
- Convert low-cardinality strings to `category` (integer dictionary codes).
- Downcast `Score` to `int16` (range: $-32{,}768$ to $32{,}767$).

In [ ]:
df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').astype(np.float32)
df['State'] = df['State'].astype('category')
df['Score'] = df['Score'].astype(np.int16)

mem_opt = df.memory_usage(deep=True).sum() / 1024
print(f"Optimized RAM:     {mem_opt:.1f} KB ({100 * (1 - mem_opt / mem_raw):.1f}% memory reduction)")

### The Split-Apply-Combine Pattern (`groupby`)
We aggregate transaction rows into customer-level behavioral predictors.

In [ ]:
customer_features = df.groupby('CustomerID').agg(
    total_spend=('Amount', 'sum'),
    avg_spend=('Amount', 'mean'),
    tx_count=('Amount', 'count'),
    credit_score=('Score', 'first')
).reset_index()

customer_features.head(3)

### Production Dummy Schema Alignment
If a categorical level appears in training but is absent in testing (or vice versa), naive `pd.get_dummies` creates mismatched column dimensions. 
We align the test split against the training schema using `.reindex()`.

In [ ]:
df_train = pd.DataFrame({'State': ['CA', 'NY', 'TX', 'FL']})
df_test  = pd.DataFrame({'State': ['CA', 'NY', 'TX']})  # FL missing in test!

train_enc = pd.get_dummies(df_train['State'], drop_first=True)
test_enc  = pd.get_dummies(df_test['State'], drop_first=True)

# Reindex test against train columns, filling unseen categories with 0
test_aligned = test_enc.reindex(columns=train_enc.columns, fill_value=0)

assert list(train_enc.columns) == list(test_aligned.columns)
print("Aligned dummy schema:")
print(test_aligned)